# Notebook da Camada Bronze

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
volume_path = f"/Volumes/{catalog}/bronze/landing"

# Tabelas Bronze a partir dos CSVs
tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]



In [0]:
# Loop de criação das tabelas
for tabela in tabelas:
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(f"{volume_path}/*{tabela}*.csv")
          .withColumn("ingestion_datetime", F.current_timestamp()))


    (df.write     # escreve a tabela como delta em modo append
       .format("delta")
       .mode("append")
       .saveAsTable(f"{bronze_schema}.{tabela}"))

In [0]:
#numero de rows em cada tabela
tabelas = spark.catalog.listTables(bronze_schema)
for t in tabelas:
  print(f"{t.name}: {spark.table(f'{bronze_schema}.{t.name}').count()} linhas")

In [0]:
# Celula adicional mostrando todas as tabelas da bronze carregadas (serve de apoio para as transformações feitas na camada silver)

for t in tabelas:
    display(spark.table(f"{bronze_schema}.{t.name}"))      